In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway
import logging
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from sklearn.decomposition import NMF

In [7]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [8]:
# Scalable Data Loading with Dask
def load_data_scalable(file_path):
    logging.info("Loading data with Dask for scalability")
    df = dd.read_csv(file_path).compute()  # Convert to pandas after processing
    return df

In [9]:
# Data Pipeline
def process_data(df):
    logging.info("Processing data in pipeline")
    df.dropna(inplace=True)
    for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
        df[col] = df[col].astype('category')
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df['Engagement_Score'] = df[numeric_cols].mean(axis=1)
    df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))
    return df, scaler

In [18]:
# Advanced NLP Features
def add_nlp_features(df):
    logging.info("Adding advanced NLP features")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['Hashtag'].tolist(), show_progress_bar=True)
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['Hashtag_Cluster'] = kmeans.fit_predict(embeddings)
    cluster_names = {0: 'Trendy', 1: 'Informative', 2: 'Casual'}
    df['Cluster_Name'] = df['Hashtag_Cluster'].map(cluster_names)
    
    sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    df['Hashtag_Sentiment'] = df['Hashtag'].apply(
        lambda x: sentiment_analyzer(x)[0]['score'] if sentiment_analyzer(x)[0]['label'] == 'POSITIVE' else -sentiment_analyzer(x)[0]['score']
    )
    
    from sklearn.decomposition import NMF
    nmf = NMF(n_components=3, random_state=42)
    embeddings_non_negative = embeddings - embeddings.min()  # Shift to non-negative
    topic_weights = nmf.fit_transform(embeddings_non_negative)
    df['Dominant_Topic'] = topic_weights.argmax(axis=1)
    topic_names = {0: 'Entertainment', 1: 'Education', 2: 'Lifestyle'}
    df['Topic_Name'] = df['Dominant_Topic'].map(topic_names)
    
    top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
    df['Top_Hashtag_Platform'] = df['Platform'].map(top_hashtags)
    return df

In [19]:
# Predictive Modeling with Interpretability
def train_predictive_model(df):
    logging.info("Training XGBoost model with SHAP interpretability")
    features = ['Views', 'Likes', 'Shares', 'Comments', 'Hashtag_Sentiment', 'Hashtag_Cluster', 'Dominant_Topic']
    X = df[features]
    y = df['Engagement_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # SHAP Explainability
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.savefig("shap_summary.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    logging.info(f"Model Performance - MSE: {mse:.4f}, R2: {r2:.4f}")
    dump(xgb_model, 'xgb_model.joblib')
    logging.info("Model saved as 'xgb_model.joblib'")
    return xgb_model, mse, r2

In [20]:
# Visualization Generation
def generate_visualizations(df):
    logging.info("Generating visualizations")
    sns.set_style("whitegrid")
    
    # Bar Chart
    platform_eng = df.groupby('Platform').agg({'Engagement_Score': 'mean'}).reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=platform_eng, x='Platform', y='Engagement_Score', palette='colorblind')
    plt.title("Engagement by Platform", fontsize=16)
    plt.savefig("engagement_by_platform.png", dpi=300)
    plt.close()

    # Pie Chart
    content_dist = df['Content_Type'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(content_dist, labels=content_dist.index, autopct='%1.1f%%', colors=sns.color_palette('colorblind', n_colors=len(content_dist)))
    plt.title("Content Type Distribution", fontsize=16)
    plt.savefig("content_type_distribution.png", dpi=300)
    plt.close()

    # Grouped Bar Chart
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Engagement_Level', y='Hashtag_Sentiment', hue='Platform', palette='colorblind')
    plt.title("Sentiment by Engagement Level", fontsize=16)
    plt.savefig("sentiment_by_engagement.png", dpi=300)
    plt.close()

    # Line Chart
    df['Index'] = range(len(df))
    plt.figure(figsize=(12, 6))
    plt.plot(df['Index'], df['Engagement_Score'], label='Actual', color=sns.color_palette('colorblind')[0])
    plt.plot(df['Index'], df['Trend_Prediction'], label='Predicted', color=sns.color_palette('colorblind')[1])
    plt.title("Engagement Trend", fontsize=16)
    plt.legend()
    plt.savefig("engagement_trend.png", dpi=300)
    plt.close()

    # Heatmap
    plt.figure(figsize=(10, 6))
    pivot = df.pivot_table(values='Engagement_Score', index='Region', columns='Cluster_Name', aggfunc='mean')
    sns.heatmap(pivot, cmap='YlGnBu', annot=True, fmt='.2f', cbar_kws={'label': 'Engagement Score'})
    plt.title("Engagement by Region & Cluster", fontsize=16)
    plt.savefig("engagement_heatmap.png", dpi=300)
    plt.close()

In [ ]:
# Main Execution
logging.info("Starting OpenAI-inspired EDA pipeline")
df = load_data_scalable("Viral_Social_Media_Trends.csv")
df, scaler = process_data(df)
df = add_nlp_features(df)
xgb_model, mse, r2 = train_predictive_model(df)
df['Trend_Prediction'] = xgb_model.predict(df[['Views', 'Likes', 'Shares', 'Comments', 'Hashtag_Sentiment', 'Hashtag_Cluster', 'Dominant_Topic']])
generate_visualizations(df)
